# Final Project

## Architectural Decision: Infrastructure & Security Compliance

### Context

The project requires a high-performance vector database to support hybrid search capabilities within a RAG (Retrieval-Augmented Generation) pipeline. While managed cloud solutions are available, modern enterprise standards increasingly mandate that sensitive data processing remains strictly within the organization's protected security perimeter.

### Decision

In light of these requirements, this project prioritizes the deployment of Qdrant as a locally hosted container (simulating deployment within a corporate internal infrastructure) rather than utilizing a managed cloud service.

### Rationale

**Data Sovereignty:** By keeping the vector store and the RAG retrieval process within the corporate network, we ensure that all sensitive data remains strictly within the company's information security perimeter.

**Regulatory Compliance:** This approach aligns with internal policies that prohibit the transmission of proprietary or sensitive data to third-party cloud environments. This is often a decisive factor for maintaining the integrity of the knowledge base and is a key consideration during software selection.

**Security Control:** Local deployment allows for direct oversight of access logs, network isolation, and encryption protocols in accordance with the security standards of most organizations.

## Dataset

### Dataset Selection Rationale

#### Context

To fully implement and validate the concepts covered in the Qdrant Essentials course, the ["Global News Dataset"](https://www.kaggle.com/datasets/everydaycodings/global-news-dataset) (available via Kaggle) was selected as the primary testing resource.

#### Selection Criteria

**Content Variability:** The dataset features a combination of concise news briefs and extensive long-form articles. This variety is essential for implementing and testing different chunking strategies, ensuring that the retrieval process effectively captures both the broad context and specific details.

**Hybrid Search Implementation:** The availability of rich metadata (author, publication, date) alongside extensive body text allows for a comprehensive implementation of Hybrid Search. This enables the combination of dense vector similarity with sparse vector retrieval and metadata filtering.

**RAG Pipeline Validation:** The scale and thematic breadth of the dataset provide an ideal environment for building a robust Retrieval-Augmented Generation (RAG) pipeline, where the quality of context retrieval is critical for the LLM’s performance.

**Performance Benchmarking:** The dataset size is sufficient to demonstrate Qdrant’s indexing efficiency and its ability to handle high-dimensional vector embeddings in a scenario that closely mimics real-world production environments.

Load the selected dataset:

In [21]:
from kagglehub import dataset_load, KaggleDatasetAdapter

dataset = dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "everydaycodings/global-news-dataset",
    "data.csv",
)

dataset.head(5)

,article_id,source_id,source_name,author,title,description,url,url_to_image,published_at,content,category,full_content
0,89541,NaN,International Business Times,Paavan MATHEMA,UN Chief Urges World To 'Stop The Madness' Of ...,UN Secretary-General Antonio Guterres urged th...,https://www.ibtimes.com/un-chief-urges-world-s...,https://d.ibtimes.com/en/full/4496078/nepals-g...,2023-10-30 10:12:35.000000,UN Secretary-General Antonio Guterres urged th...,Nepal,UN Secretary-General Antonio Guterres urged th...
1,89542,NaN,Prtimes.jp,NaN,RANDEBOOよりワンランク上の大人っぽさが漂うニットとベストが新登場。,[株式会社Ainer]\nRANDEBOO（ランデブー）では2023年7月18日(火)より公...,https://prtimes.jp/main/html/rd/p/000000147.00...,https://prtimes.jp/i/32220/147/ogp/d32220-147-...,2023-10-06 04:40:02.000000,"RANDEBOO2023718()WEB2023 Autumn Winter \n""Nepa...",Nepal,NaN
2,89543,NaN,VOA News,webdesk@voanews.com (Agence France-Presse),UN Chief Urges World to 'Stop the Madness' of ...,UN Secretary-General Antonio Guterres urged th...,https://www.voanews.com/a/un-chief-urges-world...,https://gdb.voanews.com/01000000-0a00-0242-60f...,2023-10-30 10:53:30.000000,"Kathmandu, Nepal UN Secretary-General Antonio...",Nepal,NaN
3,89545,NaN,The Indian Express,Editorial,Sikkim warning: Hydroelectricity push must be ...,Ecologists caution against the adverse effects...,https://indianexpress.com/article/opinion/edit...,https://images.indianexpress.com/2023/10/edit-...,2023-10-06 01:20:24.000000,At least 14 persons lost their lives and more ...,Nepal,At least 14 persons lost their lives and more ...
4,89547,NaN,The Times of Israel,Jacob Magid,"200 foreigners, dual nationals cut down in Ham...","France lost 35 citizens, Thailand 33, US 31, U...",https://www.timesofisrael.com/200-foreigners-d...,https://static.timesofisrael.com/www/uploads/2...,2023-10-27 01:08:34.000000,"Scores of foreign citizens were killed, taken ...",Nepal,NaN


### Dataset cleaning

In modern RAG (Retrieval-Augmented Generation) architectures, the quality of indexed data directly determines system accuracy. Thorough data cleaning and normalization address several critical challenges:
- **Improving Semantic Accuracy**: Embedding models are highly sensitive to noise. Removing redundant characters, tags, and formatting artifacts minimizes vector distortion, ensuring that document proximity in the latent space is based on meaning rather than formatting quirks.
- **Mitigating Hallucinations**: High-quality cleaning ensures reliable grounding for LLM responses. Excluding "garbage" data from the context allows the model to operate strictly on relevant facts.
- **Infrastructure Optimization**: Deduplication and the removal of redundant fields reduce the size of the vector index. This improves search latency and decreases storage costs within the Vector DB.
- **Filter Reliability**: Normalizing metadata (such as case folding and standardizing date formats like RFC 3339) ensures consistency when using hybrid search and hard filters.

Summary: An effective data preparation pipeline transforms a raw dataset into a structured knowledge base optimized for high-precision retrieval.

#### Preliminary dataset cleaning

The following fields have been selected as metadata:
- `source_name` — source title.
- `published_at` — publication date.
- `category` — news category.

The `title` field will be merged into `full_content`, as titles provide critical semantic context. The updated `full_content` field will serve as the primary source for vector search.
The `author` and `url_to_image` fields are discarded: the former contains significant noise, while the latter often points to unreachable resources.

In [22]:
import pandas as pd

dataset = dataset.drop(columns=['article_id', 'source_id', 'content', 'description', 'url_to_image', 'author'])

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30 10:12:35.000000,Nepal,UN Secretary-General Antonio Guterres urged th...
1,Prtimes.jp,RANDEBOOよりワンランク上の大人っぽさが漂うニットとベストが新登場。,https://prtimes.jp/main/html/rd/p/000000147.00...,2023-10-06 04:40:02.000000,Nepal,NaN
2,VOA News,UN Chief Urges World to 'Stop the Madness' of ...,https://www.voanews.com/a/un-chief-urges-world...,2023-10-30 10:53:30.000000,Nepal,NaN
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06 01:20:24.000000,Nepal,At least 14 persons lost their lives and more ...
4,The Times of Israel,"200 foreigners, dual nationals cut down in Ham...",https://www.timesofisrael.com/200-foreigners-d...,2023-10-27 01:08:34.000000,Nepal,NaN
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29 10:57:22,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29 08:41:18,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29 10:01:12,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29 13:44:33,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


Drop missing values from the dataset:

In [23]:
dataset = dataset.dropna(subset=['source_name', 'title', 'url', 'published_at', 'category', 'full_content'])

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30 10:12:35.000000,Nepal,UN Secretary-General Antonio Guterres urged th...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06 01:20:24.000000,Nepal,At least 14 persons lost their lives and more ...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25 09:58:17.000000,Nepal,"India, the first non-Arab country to recognise..."
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02 05:48:58.000000,Nepal,Written by Alex Travelli and Hari Kumar No nat...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02 01:12:47.000000,Nepal,NEW DELHI: India preferred Bangladesh over Nep...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29 10:57:22,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29 08:41:18,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29 10:01:12,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29 13:44:33,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


Strip newlines and leading/trailing whitespace from all fields:

In [24]:
metadata_columns = ['source_name', 'title', 'category']

for col in metadata_columns:
    dataset[col] = dataset[col].str.replace(r'[\n\r]+', ' ', regex=True)
    dataset[col] = dataset[col].str.strip()

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30 10:12:35.000000,Nepal,UN Secretary-General Antonio Guterres urged th...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06 01:20:24.000000,Nepal,At least 14 persons lost their lives and more ...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25 09:58:17.000000,Nepal,"India, the first non-Arab country to recognise..."
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02 05:48:58.000000,Nepal,Written by Alex Travelli and Hari Kumar No nat...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02 01:12:47.000000,Nepal,NEW DELHI: India preferred Bangladesh over Nep...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29 10:57:22,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29 08:41:18,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29 10:01:12,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29 13:44:33,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


Deduplicate rows:

In [25]:
dataset = dataset.drop_duplicates()

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30 10:12:35.000000,Nepal,UN Secretary-General Antonio Guterres urged th...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06 01:20:24.000000,Nepal,At least 14 persons lost their lives and more ...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25 09:58:17.000000,Nepal,"India, the first non-Arab country to recognise..."
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02 05:48:58.000000,Nepal,Written by Alex Travelli and Hari Kumar No nat...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02 01:12:47.000000,Nepal,NEW DELHI: India preferred Bangladesh over Nep...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29 10:57:22,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29 08:41:18,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29 10:01:12,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29 13:44:33,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


#### Attribute `source_name`

In [26]:
for word in sorted(dataset['source_name'].unique()):
    print(word)

ABC News
Al Jazeera English
AllAfrica - Top Africa News
Android Central
BBC News
Boing Boing
Business Insider
CNA
CNN
Deadline
Digital Trends
ETF Daily News
Euronews
Forbes
Gizmodo.com
Globalsecurity.org
GlobeNewswire
International Business Times
Marketscreener.com
NPR
Phys.Org
RT
ReadWrite
The Indian Express
The Punch
The Times of India
The Verge
Time
Wired


No further issues in this field.

#### Attribute `category`

Targeted replacements to resolve inconsistencies within the same category:

In [27]:
dataset['category'] = dataset['category'].str.replace('America', 'United States', regex=False)

dataset['category'] = dataset['category'].str.replace('Congo, The Democratic Republic of the', 'Congo', regex=False)
dataset['category'] = dataset['category'].str.replace('Iran, Islamic Republic of', 'Iran', regex=False)
dataset['category'] = dataset['category'].str.replace('Korea, Republic of', 'Korea', regex=False)
dataset['category'] = dataset['category'].str.replace('Taiwan, Province of China', 'Taiwan', regex=False)
dataset['category'] = dataset['category'].str.replace('world', 'World', regex=False)

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30 10:12:35.000000,Nepal,UN Secretary-General Antonio Guterres urged th...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06 01:20:24.000000,Nepal,At least 14 persons lost their lives and more ...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25 09:58:17.000000,Nepal,"India, the first non-Arab country to recognise..."
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02 05:48:58.000000,Nepal,Written by Alex Travelli and Hari Kumar No nat...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02 01:12:47.000000,Nepal,NEW DELHI: India preferred Bangladesh over Nep...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29 10:57:22,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29 08:41:18,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29 10:01:12,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29 13:44:33,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


Checking unique values:

In [28]:
for word in sorted(dataset['category'].unique()):
    print(word)

Afghanistan
Africa
Albania
Algeria
Amazon
Andorra
Angola
Anime
Antarctica
Architecture
Argentina
Armenia
Art
Artificial Intelligence
Aruba
Asia
Astronomy
Australia
Austria
Azerbaijan
Bahamas
Bahrain
Bangladesh
Barbados
Beauty
Belarus
Belgium
Belize
Benin
Bermuda
Bhutan
Bitcoin
Blockchain
Bosnia and Herzegovina
Botswana
Brazil
Bulgaria
Burkina Faso
Burundi
COVID
Cabo Verde
Cambodia
Cameroon
Canada
Cars
Cayman Islands
Central African Republic
Chad
Chile
China
Christmas Island
Climate
Coding
Colombia
Congo
Costa Rica
Croatia
Cryptocurrency
Cuba
Cyprus
Côte d'Ivoire
DIY
Denmark
Design
Djibouti
Dominican Republic
Ecuador
Education
Egypt
El Salvador
Entrepreneurship
Eritrea
Estonia
Ethiopia
Europe
Facebook
Fashion
Fiji
Finance
Finland
Fitness
Food
France
Gabon
Gambia
Games
Gardening
Georgia
Germany
Ghana
Gibraltar
Google
Greece
Greenland
Guam
Guatemala
Guernsey
Guinea
Guyana
Haiti
Happiness
Health
Hiking
History
Home
Honduras
Hong Kong
Hungary
Iceland
India
Indonesia
Instagram
Iran
Iraq
Irel

There are no more issues here either.

#### Attribute `published_at`

Format the published_at field as RFC 3339 (ISO 8601 string):

In [29]:
import pandas as pd

dataset['published_at'] = pd.to_datetime(dataset['published_at'], format='mixed')
dataset['published_at'] = dataset['published_at'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,Nepal,UN Secretary-General Antonio Guterres urged th...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06T01:20:24Z,Nepal,At least 14 persons lost their lives and more ...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25T09:58:17Z,Nepal,"India, the first non-Arab country to recognise..."
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02T05:48:58Z,Nepal,Written by Alex Travelli and Hari Kumar No nat...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02T01:12:47Z,Nepal,NEW DELHI: India preferred Bangladesh over Nep...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29T10:57:22Z,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29T08:41:18Z,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29T10:01:12Z,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29T13:44:33Z,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


#### Attribute `title`

We will remove all rows where the `title` length exceeds the length of the `full_content` text:

In [30]:
dataset = dataset[dataset['title'].str.len() < dataset['full_content'].str.len()]

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,Nepal,UN Secretary-General Antonio Guterres urged th...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06T01:20:24Z,Nepal,At least 14 persons lost their lives and more ...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25T09:58:17Z,Nepal,"India, the first non-Arab country to recognise..."
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02T05:48:58Z,Nepal,Written by Alex Travelli and Hari Kumar No nat...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02T01:12:47Z,Nepal,NEW DELHI: India preferred Bangladesh over Nep...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29T10:57:22Z,Home,Karnataka Deputy Chief Minister D K Shivakumar...
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29T08:41:18Z,Home,FC Barcelona have guaranteed at least $767.6 m...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29T10:01:12Z,Home,The photo from David and Sarah Lubarsky's wedd...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29T13:44:33Z,Home,Kerber’s Farm: Bringing Farm To Table To Manha...


The `title` usually captures the essence of the article, and its presence within the text can enhance semantic search in certain cases. We will merge the `title` with the main body using a structural highlighting method.

In [31]:
dataset['full_content'] = (
    "Title: " + dataset['title'] + 
    ". Content: " + dataset['full_content']
)

dataset

,source_name,title,url,published_at,category,full_content
0,International Business Times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,Nepal,Title: UN Chief Urges World To 'Stop The Madne...
3,The Indian Express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06T01:20:24Z,Nepal,Title: Sikkim warning: Hydroelectricity push m...
6,Al Jazeera English,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25T09:58:17Z,Nepal,Title: Pro-Israel rallies allowed in India but...
7,The Indian Express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02T05:48:58Z,Nepal,Title: No nation in the world is buying more p...
12,The Times of India,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02T01:12:47Z,Nepal,Title: PM Hasina’s war on terror gets daughter...
...,...,...,...,...,...,...
105370,The Indian Express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29T10:57:22Z,Home,"Title: Have done no wrong, only did party work..."
105371,Forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29T08:41:18Z,Home,Title: FC Barcelona Guarantees $77.6 Million C...
105372,NPR,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29T10:01:12Z,Home,Title: Three hospitals ignored her gravely ill...
105373,Forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29T13:44:33Z,Home,Title: Kerber’s Farm: Bringing Farm To Table T...


The search will return a URL to the article. We will remove the `title` field, as it is no longer needed.

In [ ]:
# dataset = dataset.drop('title', axis=1)

# dataset

,source_name,url,published_at,category,full_content
0,International Business Times,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,Nepal,Title: UN Chief Urges World To 'Stop The Madne...
3,The Indian Express,https://indianexpress.com/article/opinion/edit...,2023-10-06T01:20:24Z,Nepal,Title: Sikkim warning: Hydroelectricity push m...
6,Al Jazeera English,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25T09:58:17Z,Nepal,Title: Pro-Israel rallies allowed in India but...
7,The Indian Express,https://indianexpress.com/article/business/avi...,2023-11-02T05:48:58Z,Nepal,Title: No nation in the world is buying more p...
12,The Times of India,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02T01:12:47Z,Nepal,Title: PM Hasina’s war on terror gets daughter...
...,...,...,...,...,...
105370,The Indian Express,https://indianexpress.com/article/cities/banga...,2023-11-29T10:57:22Z,Home,"Title: Have done no wrong, only did party work..."
105371,Forbes,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29T08:41:18Z,Home,Title: FC Barcelona Guarantees $77.6 Million C...
105372,NPR,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29T10:01:12Z,Home,Title: Three hospitals ignored her gravely ill...
105373,Forbes,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29T13:44:33Z,Home,Title: Kerber’s Farm: Bringing Farm To Table T...


#### Normalization metadata fileds

To prevent case mismatches during metadata filtering, all fields will be lowercased. Similarly, we'll apply this in Qdrant for all string-based metadata to maintain consistency.

In [32]:
for col in ['source_name', 'category']:
    dataset[col] = dataset[col].str.lower()

dataset

,source_name,title,url,published_at,category,full_content
0,international business times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,nepal,Title: UN Chief Urges World To 'Stop The Madne...
3,the indian express,Sikkim warning: Hydroelectricity push must be ...,https://indianexpress.com/article/opinion/edit...,2023-10-06T01:20:24Z,nepal,Title: Sikkim warning: Hydroelectricity push m...
6,al jazeera english,Pro-Israel rallies allowed in India but Palest...,https://www.aljazeera.com/news/2023/10/25/pro-...,2023-10-25T09:58:17Z,nepal,Title: Pro-Israel rallies allowed in India but...
7,the indian express,No nation in the world is buying more planes t...,https://indianexpress.com/article/business/avi...,2023-11-02T05:48:58Z,nepal,Title: No nation in the world is buying more p...
12,the times of india,PM Hasina’s war on terror gets daughter India’...,https://timesofindia.indiatimes.com/india/pm-h...,2023-11-02T01:12:47Z,nepal,Title: PM Hasina’s war on terror gets daughter...
...,...,...,...,...,...,...
105370,the indian express,"Have done no wrong, only did party work, says ...",https://indianexpress.com/article/cities/banga...,2023-11-29T10:57:22Z,home,"Title: Have done no wrong, only did party work..."
105371,forbes,FC Barcelona Guarantees $77.6 Million Champion...,https://www.forbes.com/sites/tomsanderson/2023...,2023-11-29T08:41:18Z,home,Title: FC Barcelona Guarantees $77.6 Million C...
105372,npr,Three hospitals ignored her gravely ill fiancé...,https://www.npr.org/2023/11/29/1215016001/heal...,2023-11-29T10:01:12Z,home,Title: Three hospitals ignored her gravely ill...
105373,forbes,Kerber’s Farm: Bringing Farm To Table To Manha...,https://www.forbes.com/sites/garystern/2023/11...,2023-11-29T13:44:33Z,home,Title: Kerber’s Farm: Bringing Farm To Table T...


Save the cleaned dataset:

In [33]:
dataset.to_parquet("dataset.parquet.gzip", compression="gzip")

## Additional functions

Function for measuring search delays:

In [66]:
import time
import requests
import pandas as pd
from typing import Callable

BENCHMARK_COLS = [
    "latency_mean_ms",
    "latency_p50_ms",
    "latency_p90_ms",
    "latency_p95_ms",
    "latency_p99_ms",
]

latency_results = pd.DataFrame(columns=BENCHMARK_COLS)


def benchmark_search(fn: Callable, result_df: pd.DataFrame, test_name: str, warmup_runs: int = 3, n_runs: int = 20) -> pd.DataFrame:
    
    rows = []

    # Warmup search
    for _ in range(warmup_runs):
        fn()

    # Measurements
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        latency_ms = (time.perf_counter() - t0) * 1000
        rows.append(latency_ms)

    # Calculate latency metrics
    stats = pd.Series(rows)
    metrics = stats.describe(percentiles=[0.5, 0.9, 0.95, 0.99]) #.round(2)
    
    # Update results
    result_df.loc[test_name] = [
        metrics["mean"],     # latency_mean_ms
        metrics["50%"],      # latency_p50_ms
        metrics["90%"],      # latency_p90_ms
        metrics["95%"],      # latency_p95_ms
        metrics["99%"]       # latency_p99_ms
    ]

    return result_df


def google_search(query: str) -> str:
    return requests.get("https://www.google.com/search", params={"q": query}, timeout=10).text

def duckduckgo_search(query: str) -> str:
    return requests.get("https://html.duckduckgo.com/html/", params={"q": query}, timeout=10).text


benchmark_search(lambda: google_search("python pandas"), latency_results, "Google search")
benchmark_search(lambda: duckduckgo_search("python pandas"), latency_results, "DuckDuckGo search")

latency_results

,latency_mean_ms,latency_p50_ms,latency_p90_ms,latency_p95_ms,latency_p99_ms
Google search,578.534937,549.386218,623.097373,699.069694,1072.207576
DuckDuckGo search,395.736543,398.374213,413.218756,418.955833,420.266133


A function for measuring ranking and search metrics:

In [67]:
import pandas as pd
from ranx import Qrels, Run, evaluate

# Ваши столбцы
metrics = ["mrr", "recall@10", "ndcg@10"]
metrics_results = pd.DataFrame(columns=metrics)

def calculate_search_metrics(predicted, ground_truth, metrics, test_name, results_df):
    query_id = "test_query"
    
    # Ground Truth with weghts
    gt_data = Qrels({
        query_id: {
            doc_id: len(ground_truth) - i 
            for i, doc_id in enumerate(ground_truth)
        }
    })
    
    # Search results with weights
    pd_data = Run({
        query_id: {
            doc_id: len(predicted) - i 
            for i, doc_id in enumerate(predicted)
        }
    })
    
    # Calculating metrics
    scores = evaluate(gt_data, pd_data, metrics)
    
    # Updating DataFrame with results
    results_df.loc[test_name] = [scores.get(col, 0.0) for col in metrics]


truth = ["doc_A", "doc_B", "doc_C"]

# Test 1: Perfect order (the same as in truth)
calculate_search_metrics(["doc_A", "doc_B", "doc_C"], truth, metrics, "perfect_order", metrics_results)

# Test 2: The same documents, but in reverse order
calculate_search_metrics(["doc_C", "doc_B", "doc_A"], truth, metrics, "reversed_order", metrics_results)

# Test 3: The same documents, but the first one (most important) is missing
calculate_search_metrics(["doc_B", "doc_C"], truth, metrics, "missing_top_1", metrics_results)

print(metrics_results)

                mrr  recall@10   ndcg@10
perfect_order   1.0   1.000000  1.000000
reversed_order  1.0   1.000000  0.789998
missing_top_1   1.0   0.666667  0.552500


Функция для подготовки данных перед загрузкой с учётом определенного сплиттера:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import pandas as pd

encoder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30,
    )

def dataset_stream(df, splitter, gpu_batch_size=64):
    meta_cols = ["source_name", "title", "url", "published_at", "category"]
    
    batch_texts = []
    batch_payloads = []
    
    for row in df.itertuples(index=False):
        # Базовые метаданные текущей строки
        payload_base = {col: getattr(row, col) for col in meta_cols}
        
        # Режем текст переданным сплиттером
        chunks = splitter.split_text(str(row.full_content))
        
        for chunk in chunks:
            batch_texts.append(f"passage: {chunk}")
            # Добавляем сам чанк в метаданные, чтобы видеть его при поиске
            current_payload = payload_base.copy()
            current_payload["text"] = chunk
            batch_payloads.append(current_payload)
            
            # Если набрали батч для GPU — кодируем и отдаем
            if len(batch_texts) == gpu_batch_size:
                vectors = encoder.encode(batch_texts, convert_to_numpy=True)
                for v, p in zip(vectors, batch_payloads):
                    yield v, p
                batch_texts, batch_payloads = [], []
                
    # Sending the remaining items
    if batch_texts:
        vectors = encoder.encode(batch_texts, convert_to_numpy=True)
        for v, p in zip(vectors, batch_payloads):
            yield v, p

test_df = pd.read_parquet("dataset.parquet.gzip")[:128]

dataset_streamer = dataset_stream(test_df, text_splitter)

embeddings, payloads = next(dataset_streamer)

print(embeddings.shape, payloads)

del encoder, text_splitter, dataset_streamer, test_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(768,) {'source_name': 'international business times', 'title': "UN Chief Urges World To 'Stop The Madness' Of Climate Change", 'url': 'https://www.ibtimes.com/un-chief-urges-world-stop-madness-climate-change-3716939', 'published_at': '2023-10-30T10:12:35Z', 'category': 'nepal', 'text': 'Title: UN Chief Urges World To \'Stop The Madness\' Of Climate Change. Content: UN Secretary-General Antonio Guterres urged the world Monday to "stop the madness" of climate change as he visited Himalayan regions struggling from rapidly melting glaciers to witness the devastating impact of the'}


В некоторых случаях можно сразу подготовить все данные и просто загружать их методом `client.upload_collection()`.

## Setting Up the Development Environment

### Подключение к хранилищу

Connect to the local Qdrant storage:

> [Access to the Qdrant dashboard from Docker host](http://localhost:6333/dashboard)

In [5]:
from qdrant_client import QdrantClient, models
import os

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

collections = client.get_collections()
print(f"Connected to Qdrant Cloud: {len(collections.collections)} collections")

Connected to Qdrant Cloud: 1 collections


/tmp/ipykernel_685/396165620.py:4: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))


### Выбор модели для плотных векторов

В качестве модели для создания Dense Vectors эмбеддингов выберем `multilingual-e5-base`

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

encoder = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base", model_kwargs={"device": "cuda"})

text = "Привет, мир!"
vector = encoder.embed_query(text)

print(f"Vector length: {len(vector)}, \n {vector}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector length: 768, 
 [0.02919846773147583, 0.04641994461417198, -0.015342864207923412, 0.04352734982967377, 0.005529590882360935, -0.041299235075712204, -0.018318360671401024, -0.008249619975686073, 0.019141297787427902, 0.036783359944820404, -0.02454104833304882, -0.004741741809993982, 0.19188784062862396, 0.04812153801321983, -0.03860395774245262, -0.06411952525377274, -0.0035166272427886724, -0.030167503282427788, 0.004394894931465387, 0.0036365268751978874, 0.05462142825126648, -0.02209370583295822, 0.04198655113577843, -0.0461546927690506, 0.04129008203744888, -0.01138180959969759, 0.021426254883408546, 0.006927143782377243, -0.018990570679306984, 0.012140760198235512, 0.02302173711359501, -0.05457295849919319, 0.027344875037670135, 0.007848447188735008, 0.005846124142408371, 0.044427674263715744, -0.008338356390595436, -0.02434696815907955, 0.04735226929187775, 0.004183465614914894, 0.028512168675661087, 0.031187385320663452, -0.011971750296652317, -0.008447249419987202, 0.03649

## Плотные вектора - стратегия фрагментации текста

### Выбор метрики для плотных векторов

В качестве метрики выбрано косинусное сходство (Cosine Similarity), так как модели из семейства E5 обучались с использованием функции потерь, которая оптимизирует именно косинусное расстояние. Это значит, что семантическая близость текстов в векторном пространстве модели напрямую соответствует углу между векторами.

Кроме того, косинусное сходство учитывает только направление векторов, игнорируя их длину (норму). Если один и тот же смысл выражен кратко или чуть более развернуто, косинусное сходство сочтет их близкими.

### Выбор оптимальной страгении фрагментации текста

Выбор стратегии чанкинга (разбиения текста) — это 50% успеха в RAG. В 2026 году, когда модели вроде multilingual-e5-base стали стандартом, подход к подготовке данных сильно сместился в сторону сохранения семантической целостности.

Кратко вспомним какие стратегии чанкинга существуют и оценим насколько они эффективны:
- Fixed-Size Chunking - это самый примитивный метод. Он часто разрывает предложения на полуслове, из-за чего теряется смысл. Нет смысла рассматривать.
- Sentence-Based Chunking - одиночное предложение часто не содержит достаточно контекста для полноценного ответа. Вектор одного предложения в E5 может быть слишком специфичным и не поймать общую тему документа. Не рассматриваем.
- Sliding Window Chunking - Чанки фиксированного размера, но с перекрытием (overlap), например, 512 токенов с нахлестом в 50-100 токенов. Перекрытие гарантирует, что контекст, который мог быть разорван на границе чанков, сохранится хотя бы в одном из них. Для RAG это критически важно для связности ответов. Но в 2026 году нет смысла использовать Sliding Window, если рассматривается Recursive Character Text Splitting.
- Recursive Character Text Splitting - пытается разбить текст по абзацам, если абзац слишком велик — по предложениям, если и они велики — по словам. Это "золотой стандарт". Он сохраняет структуру текста (сначала пытается оставить абзац целым) и гарантирует, что чанк впишется в max_seq_length (512 токенов для выбранной модели). Рассматриваем.
- Semantic-Aware Chunking - Текст разбивается там, где резко меняется смысл (анализируется расстояние между эмбеддингами соседних предложений). Это позволяет создавать «умные» границы, где каждый фрагмент — это законченная мысль. Для моделей семейства E5 это лучший компаньон, так как модель будет работать с идеально чистыми по смыслу кусками. Рассматриваем.

#### Получение тестовой выборки

Проверять каждую стратегию чанкинга на всем датасете слишком затратно по рессурсам, возьмём фрагмент датасета размером в 1500 новостных статей методом стратифицированной выборки:

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split

dataset = pd.read_parquet("dataset.parquet.gzip")

sample_size = 300 # 1500

samples, _ = train_test_split(
    dataset,
    train_size=(sample_size / len(dataset)),
    stratify=dataset['category'],  # Ключевой параметр для баланса категорий
    random_state=42           # Фиксируем результат для воспроизводимости
)

samples

,source_name,title,url,published_at,category,full_content
60932,etf daily news,New York State Common Retirement Fund Trims St...,https://www.etfdailynews.com/2023/11/05/new-yo...,2023-11-05T12:39:26Z,stock,Title: New York State Common Retirement Fund T...
104902,globalsecurity.org,Head of Tibet's government-in-exile meets with...,https://www.globalsecurity.org/military/librar...,2023-11-29T07:27:22Z,canada,Title: Head of Tibet's government-in-exile mee...
25159,wired,Graphcore Was the UK's AI Champion—Now It’s Sc...,https://www.wired.com/story/graphcore-uk-ai-ch...,2023-10-06T11:59:00Z,technology,Title: Graphcore Was the UK's AI Champion—Now ...
46395,rt,US breaks $6 billion promise to Iran,https://www.rt.com/news/584792-us-iran-frozen-...,2023-10-13T01:55:40Z,iran,Title: US breaks $6 billion promise to Iran. C...
14951,rt,Russian Communists want gay ballet icon Nureye...,https://www.rt.com/russia/585887-russia-commun...,2023-10-26T16:21:56Z,russian federation,Title: Russian Communists want gay ballet icon...
...,...,...,...,...,...,...
64448,etf daily news,Concord Wealth Partners Raises Stock Holdings ...,https://www.etfdailynews.com/2023/11/07/concor...,2023-11-07T14:02:44Z,stock,Title: Concord Wealth Partners Raises Stock Ho...
71892,globalsecurity.org,DOD Implores Congress to Provide Ukraine Defen...,https://www.globalsecurity.org/wmd/library/new...,2023-11-10T10:05:58Z,ukraine,Title: DOD Implores Congress to Provide Ukrain...
75007,bbc news,Storm Debi yellow rain warning comes into forc...,https://www.bbc.co.uk/news/uk-scotland-north-e...,2023-11-13T10:20:42Z,weather,Title: Storm Debi yellow rain warning comes in...
81035,etf daily news,So-Young International (NASDAQ:SY) versus Mara...,https://www.etfdailynews.com/2023/11/14/so-you...,2023-11-14T15:20:45Z,finance,Title: So-Young International (NASDAQ:SY) vers...


Возьмём одну статью для тестирования методов чанкинга:

In [15]:
article = [dataset.loc[74616, 'full_content']]
article

['Title: Xponential Fitness (NYSE:XPOF) Price Target Lowered to $31.00 at Raymond James. Content: Xponential Fitness (NYSE:XPOF–Free Report)had its price target trimmed by Raymond James from $40.00 to $31.00 in a report issued on Wednesday,Benzingareports. The brokerage currently has a strong-buy rating on the stock. XPOF has been the topic of several other reports. Piper Sandler dropped their price objective on Xponential Fitness from $26.00 to $20.00 in a research report on Thursday, October 12th. Stifel Nicolaus initiated coverage on Xponential Fitness in a research report on Friday, October 13th. They set a hold rating and a $18.00 price objective on the stock. Citigroup dropped their price objective on Xponential Fitness from $39.00 to $30.00 in a research report on Friday, August 4th. Guggenheim dropped their price objective on Xponential Fitness from $32.00 to $30.00 and set a buy rating on the stock in a research report on Friday, September 15th. Finally, Bank of America cut Xp

#### Recursive Character Text Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

current_chunk_size = 500

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=current_chunk_size,
    chunk_overlap=current_chunk_size * 0.1,
)

docs = text_splitter.create_documents(article)

chunks = [doc.page_content for doc in docs]

chunks

['Title: Xponential Fitness (NYSE:XPOF) Price Target Lowered to $31.00 at Raymond James. Content: Xponential Fitness (NYSE:XPOF–Free Report)had its price target trimmed by Raymond James from $40.00 to $31.00 in a report issued on Wednesday,Benzingareports. The brokerage currently has a strong-buy rating on the stock. XPOF has been the topic of several other reports. Piper Sandler dropped their price objective on Xponential Fitness from $26.00 to $20.00 in a research report on Thursday, October',
 '$20.00 in a research report on Thursday, October 12th. Stifel Nicolaus initiated coverage on Xponential Fitness in a research report on Friday, October 13th. They set a hold rating and a $18.00 price objective on the stock. Citigroup dropped their price objective on Xponential Fitness from $39.00 to $30.00 in a research report on Friday, August 4th. Guggenheim dropped their price objective on Xponential Fitness from $32.00 to $30.00 and set a buy rating on the stock in a research report on',


#### SemanticChunker

In [73]:
from langchain_experimental.text_splitter import SemanticChunker

text_splitter = SemanticChunker(encoder, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=80, min_chunk_size=50)

docs = text_splitter.create_documents(article)

chunks = [doc.page_content for doc in docs]
chunks

['Title: Xponential Fitness (NYSE:XPOF) Price Target Lowered to $31.00 at Raymond James. Content: Xponential Fitness (NYSE:XPOF–Free Report)had its price target trimmed by Raymond James from $40.00 to $31.00 in a report issued on Wednesday,Benzingareports. The brokerage currently has a strong-buy rating on the stock. XPOF has been the topic of several other reports. Piper Sandler dropped their price objective on Xponential Fitness from $26.00 to $20.00 in a research report on Thursday, October 12th. Stifel Nicolaus initiated coverage on Xponential Fitness in a research report on Friday, October 13th. They set a hold rating and a $18.00 price objective on the stock. Citigroup dropped their price objective on Xponential Fitness from $39.00 to $30.00 in a research report on Friday, August 4th. Guggenheim dropped their price objective on Xponential Fitness from $32.00 to $30.00 and set a buy rating on the stock in a research report on Friday, September 15th. Finally, Bank of America cut Xp

#### Создание коллекции

In [79]:
from qdrant_client.models import Distance, VectorParams

strategies = [
	{"name": "recursive_character", "chunk_size": [300, 400, 450, 500]},        # "chunk_overlap": [30, 40, 45, 50]
	{"name": "semantic", "breakpoint_threshold_amount": [70, 80, 85, 90, 95]}   #  "breakpoint_threshold_type": ["percentile"], "min_chunk_size": [100]
]

# Создание коллекции
collection_name = "semantic_search"


def get_collections_names(strategies):

    result = []

    for strategy in strategies:
        name = strategy["name"]

        if name == "recursive_character":
            for chunk_size in strategy["chunk_size"]:
                vector_name = f"{name}_{chunk_size}"
                result.append(vector_name)
            
        if name == "semantic":
            for breakpoint_threshold_amount in strategy["breakpoint_threshold_amount"]:
                vector_name = f"{name}_{breakpoint_threshold_amount}"
                result.append(vector_name)

    return result


if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

vectors_config = {}

for vector_name in get_collections_names(strategies):

    vectors_config[vector_name] = models.VectorParams(
            size=768, 
            distance=models.Distance.COSINE
            )


client.create_collection(
    collection_name=collection_name,
    vectors_config=vectors_config
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="title",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="url",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

#### Загрузка данных

In [12]:
def recursive_character_splitter(text, chunk_size):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_size * 0.1,
    )

    docs = text_splitter.create_documents(article)
    return [doc.page_content for doc in docs]

def semantic_splitter(text, threshold):
    text_splitter = SemanticChunker(
        encoder,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=threshold,
        min_chunk_size=50)

    docs = text_splitter.create_documents(article)

    return [doc.page_content for doc in docs]

In [ ]:
from tqdm.auto import tqdm

idx = 0
points = []


for news in tqdm(samples.to_dict('records'), desc="Processing news"):

    for chunk in recursive_character_splitter(news["full_content"], chunk_size=300):
        points.append(models.PointStruct(
            id=idx,
            vector={"recursive_character_300": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
        idx += 1

    for chunk in recursive_character_splitter(news["full_content"], chunk_size=400):
        points.append(models.PointStruct(
            id=idx,
            vector={"recursive_character_400": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
        idx += 1

    for chunk in recursive_character_splitter(news["full_content"], chunk_size=450):
        points.append(models.PointStruct(
            id=idx,
            vector={"recursive_character_450": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
        idx += 1

    for chunk in recursive_character_splitter(news["full_content"], chunk_size=500):
        points.append(models.PointStruct(
            id=idx,
            vector={"recursive_character_500": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
        idx += 1

    for chunk in semantic_splitter(news["full_content"], threshold=70):
        points.append(models.PointStruct(
            id=idx,
            vector={"semantic_70": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
        idx += 1

    for chunk in semantic_splitter(news["full_content"], threshold=80):
        points.append(models.PointStruct(
            id=idx,
            vector={"semantic_80": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
    idx += 1

    for chunk in semantic_splitter(news["full_content"], threshold=85):
        points.append(models.PointStruct(
            id=idx,
            vector={"semantic_85": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
    idx += 1

    for chunk in semantic_splitter(news["full_content"], threshold=90):
        points.append(models.PointStruct(
            id=idx,
            vector={"semantic_90": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
    idx += 1

    for chunk in semantic_splitter(news["full_content"], threshold=95):
        points.append(models.PointStruct(
            id=idx,
            vector={"semantic_95": encoder.embed_query(f"passage: {chunk}")},
            payload={
                "title": news["title"], 
                "url": news["url"],
                # "chunk": chunk
            }
        ))
    idx += 1
		
client.upload_points(collection_name='semantic_search', points=points)
print(f"Uploaded {idx} vectors across three chunking strategies")

Processing news:   0%|          | 0/1500 [00:00<?, ?it/s]

Uploaded 132000 vectors across three chunking strategies


#### Оценка качества поиска

In [ ]:
def search_and_compare(query, strategies, k=3):
    """Compare search results across all three chunking strategies"""
    print(f"Query: '{query}'\n")
    
    for strategy in strategies:
        results = client.query_points(
            collection_name='semantic_search',
            query=encoder.embed_query(query),
            using=strategy,
            limit=k,
        )
        
        print(f"--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            payload = point.payload
            print(f"{i}. Title: {payload['title']} | Score: {point.score:.3f}")
            print(f"   URL: {payload['url']}...")
        print()


collections = get_collections_names(strategies)

search_and_compare("Expected valuation of the global intelligent oximetry industry in the long-term business report", collections, k=10)

Query: 'Expected valuation of the global intelligent oximetry industry in the long-term business report'

--- RECURSIVE_CHARACTER_300 CHUNKING ---
1. Title: Use This Investing Formula To Reach Financial Freedom In Retirement | Score: 0.825
   URL: https://www.forbes.com/sites/bernadettejoy/2023/11/29/use-this-investing-formula-to-reach-financial-freedom-in-retirement/...
2. Title: Hamas violated international law – South Africa | Score: 0.825
   URL: https://www.rt.com/africa/587741-south-africa-condemn-hamas/...
3. Title: Wesco International Reports Third Quarter 2023 Results | Score: 0.825
   URL: https://www.marketscreener.com/quote/stock/WESCO-INTERNATIONAL-INC-14849/news/Wesco-International-Reports-Third-Quarter-2023-Results-45221006/...
4. Title: Air India gets notice over facilities for passengers | Score: 0.825
   URL: https://timesofindia.indiatimes.com/india/air-india-gets-notice-over-facilities-for-passengers/articleshow/105052982.cms...
5. Title: Govt grants approvals to 27

Целевая новость: 

Title: Smart Pulse Oximeters Market Size to Reach USD 2.3 billion by 2031 at 4.6% CAGR | Report by Transparency Market Research Inc. | Score: 0.825
URL: https://www.globenewswire.com/news-release/2023/11/07/2774725/32656/en/Smart-Pulse-Oximeters-Market-Size-to-Reach-USD-2-3-billion-by-2031-at-4-6-CAGR-Report-by-Transparency-Market-Research-Inc.html

Абсолютным победителем является стратегия RECURSIVE_CHARACTER_300 - искомая новость в 10 позиции. 

In [34]:
client.delete_collection("semantic_search")

True

### HNSW Performance Benchmarking

#### Загрузка данных

Для подбора параметров HNSW возьмём не весь датасет, а стратифицированную выборку из 5 000 статей:

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

dataset = pd.read_parquet("dataset.parquet.gzip")

sample_size = 5000

stratified_sample, _ = train_test_split(
    dataset,
    train_size=(sample_size / len(dataset)),
    stratify=dataset['category'],
    random_state=42
)

stratified_sample

,source_name,title,url,published_at,category,full_content
82201,globalsecurity.org,US secretly pushing Israel to speed up Gaza wa...,https://www.globalsecurity.org/military/librar...,2023-11-14T09:55:34Z,israel,Title: US secretly pushing Israel to speed up ...
65222,etf daily news,"Analysts Set Clearway Energy, Inc. (NYSE:CWEN)...",https://www.etfdailynews.com/2023/11/07/analys...,2023-11-07T10:59:34Z,canada,"Title: Analysts Set Clearway Energy, Inc. (NYS..."
52074,the punch,"Osun to enrol 19,000 retirees in health insura...",https://punchng.com/osun-to-enrol-19000-retire...,2023-11-03T00:15:33Z,health,"Title: Osun to enrol 19,000 retirees in health..."
84679,globenewswire,"Metabolic Testing Market To Reach USD 1,198.4 ...",https://www.globenewswire.com/news-release/202...,2023-11-16T12:10:00Z,technology,Title: Metabolic Testing Market To Reach USD 1...
73424,the times of india,Hundreds of activists demand plastic action in...,https://timesofindia.indiatimes.com/world/rest...,2023-11-11T11:11:16Z,climate,Title: Hundreds of activists demand plastic ac...
...,...,...,...,...,...,...
19506,abc news,Tropical Storm Philippe chugs toward Bermuda o...,https://abcnews.go.com/International/wireStory...,2023-10-05T12:31:50Z,"virgin islands, u.s.",Title: Tropical Storm Philippe chugs toward Be...
103680,etf daily news,"Meritage Group LP Has $233,000 Stock Holdings ...",https://www.etfdailynews.com/2023/11/27/merita...,2023-11-27T18:04:46Z,mexico,"Title: Meritage Group LP Has $233,000 Stock Ho..."
27356,business insider,The mother of tattoo artist Shani Louk who was...,https://www.businessinsider.com/mother-german-...,2023-10-08T11:20:50Z,instagram,Title: The mother of tattoo artist Shani Louk ...
49593,marketscreener.com,Dividend Payment Procedure for shareholders of...,https://www.marketscreener.com/quote/stock/AB-...,2023-10-27T11:03:01Z,lithuania,Title: Dividend Payment Procedure for sharehol...


#### Создание коллекций и подготовка данных

Создадим коллекции для подбора опитмальных параметров HNSW: 

In [ ]:
def create_collection(client, collection_name: str, m: int, ef_construct: int) -> None:
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(
            m=m,
            ef_construct=ef_construct,
            full_scan_threshold=10,
        ),
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=10
        ),
    )
    print(f"Created collection: {collection_name}")


# Test configurations
configs = [
    {"name": "fast_initial_upload", "m": 0, "ef_construct": 100},
    {"name": "memory_optimized", "m": 8, "ef_construct": 100},
    {"name": "balanced", "m": 16, "ef_construct": 200},
    {"name": "high_quality", "m": 32, "ef_construct": 400},
]

for config in configs:
    create_collection(
        client,
        collection_name=f"my_domain_{config['name']}",
        m=config["m"],
        ef_construct=config["ef_construct"],
    )

Created collection: my_domain_fast_initial_upload
Created collection: my_domain_memory_optimized
Created collection: my_domain_balanced
Created collection: my_domain_high_quality


Подготовка всех данных для загрузки:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30,
    )

def dataset_stream(df, splitter, gpu_batch_size=64):
    meta_cols = ["source_name", "title", "url", "published_at", "category"]
    
    batch_texts = []
    batch_payloads = []
    
    for row in df.itertuples(index=False):
        # Базовые метаданные текущей строки
        payload_base = {col: getattr(row, col) for col in meta_cols}
        
        # Режем текст переданным сплиттером
        chunks = splitter.split_text(str(row.full_content))
        
        for chunk in chunks:
            batch_texts.append(f"passage: {chunk}")
            # Добавляем сам чанк в метаданные, чтобы видеть его при поиске
            current_payload = payload_base.copy()
            current_payload["text"] = chunk
            batch_payloads.append(current_payload)
            
            # Если набрали батч для GPU — кодируем и отдаем
            if len(batch_texts) == gpu_batch_size:
                vectors = encoder.encode(batch_texts, convert_to_numpy=True)
                for v, p in zip(vectors, batch_payloads):
                    yield v, p
                batch_texts, batch_payloads = [], []
                
    # Sending the remaining items
    if batch_texts:
        vectors = encoder.encode(batch_texts, convert_to_numpy=True)
        for v, p in zip(vectors, batch_payloads):
            yield v, p

dataset_streamer = dataset_stream(stratified_sample, text_splitter, gpu_batch_size=128)

all_vectors, all_payloads = zip(*dataset_streamer)

#### Загрузка данных и измерениие времени

In [22]:
import time
import pandas as pd
from qdrant_client import models

def upload_with_timing(collection_name, vectors, payloads, config_name):
    # Warmup — прогрев перед замером
    if len(vectors) > 0:
        client.query_points(collection_name=collection_name, query=vectors[0], limit=1)

    # Используем perf_counter для точности
    start_time = time.perf_counter()

    client.upload_collection(
        collection_name=collection_name,
        vectors=vectors,
        payload=payloads,
        ids=None,
        batch_size=1000,
        parallel=1,
        wait=True
    )

    duration = time.perf_counter() - start_time
    print(f"{config_name}: Uploaded {len(vectors)} points in {duration:.4f}s")
    return duration

def wait_for_indexing_silent(collection_name, timeout=600):
    start_time = time.perf_counter()
    while time.perf_counter() - start_time < timeout:
        info = client.get_collection(collection_name=collection_name)
        if info.status == models.CollectionStatus.GREEN:
            return
        time.sleep(1)
    raise Exception(f"Таймаут индексации: {collection_name}")


results_list = []

for config in configs:
    name = config["name"]
    col_name = f"my_domain_{name}"
    
    duration = upload_with_timing(col_name, all_vectors, all_payloads, name)
    
    results_list.append({
        "config_name": name,
        "points_count": len(all_vectors),
        "upload_time_sec": round(duration, 4),
        "points_per_second": round(len(all_vectors) / duration, 2)
    })

    # Ожидание HNSW
    if config.get("m", 0) > 0:
        wait_for_indexing_silent(col_name)

# Превращаем результаты в DataFrame
df_results = pd.DataFrame(results_list)

print(df_results.to_string(index=False))

fast_initial_upload: Uploaded 115693 points in 19.6421s
memory_optimized: Uploaded 115693 points in 19.4942s
balanced: Uploaded 115693 points in 20.3018s
high_quality: Uploaded 115693 points in 19.0027s
        config_name  points_count  upload_time_sec  points_per_second
fast_initial_upload        115693          19.6421            5890.07
   memory_optimized        115693          19.4942            5934.73
           balanced        115693          20.3018            5698.65
       high_quality        115693          19.0027            6088.23


In [23]:
df_results

,config_name,points_count,upload_time_sec,points_per_second
0,fast_initial_upload,115693,19.6421,5890.07
1,memory_optimized,115693,19.4942,5934.73
2,balanced,115693,20.3018,5698.65
3,high_quality,115693,19.0027,6088.23


#### Benchmark Search Performance

Test search speed with different hnsw_ef values:

In [ ]:
import numpy as np

def benchmark_search(collection_name, query_embedding, query_count=100, ef_values=[64, 128, 256]):

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # hnsw_ef: higher = better recall, but slower. Tune per your latency goal.
    results = {}
    for hnsw_ef in ef_values:
        times = []

        # Run multiple queries for more reliable timing
        for _ in range(25):
            start_time = time.time()

            _ = client.query_points(
                collection_name=collection_name,
                query=query_embedding,
                limit=10,
                search_params=models.SearchParams(hnsw_ef=hnsw_ef),
                with_payload=False,
            )

            times.append((time.time() - start_time) * 1000)

        results[hnsw_ef] = {
            "avg_time": np.mean(times),
            "min_time": np.min(times),
            "max_time": np.max(times),
        }

    return results


test_query = "Smart Pulse Oximeters Market Size to Reach USD 2.3 billion by 2031 at 4.6% CAGR"
query_embedding = encoder.encode(test_query)

performance_results = {}
for config in configs:
    if config["m"] > 0:  # Skip m=0 collections for search
        collection_name = f"my_domain_{config['name']}"
        performance_results[config["name"]] = benchmark_search(
            collection_name, query_embedding
        )

performance_results

,memory_optimized,balanced,high_quality
64,"{'avg_time': 1.1963272094726562, 'min_time': 0...","{'avg_time': 1.2056446075439453, 'min_time': 1...","{'avg_time': 1.3165760040283203, 'min_time': 1..."
128,"{'avg_time': 1.335763931274414, 'min_time': 1....","{'avg_time': 1.327981948852539, 'min_time': 1....","{'avg_time': 1.5747451782226562, 'min_time': 1..."
256,"{'avg_time': 1.5015506744384766, 'min_time': 1...","{'avg_time': 1.9254207611083984, 'min_time': 1...","{'avg_time': 2.4020004272460938, 'min_time': 1..."


In [1]:
test_query = "Smart Pulse Oximeters Market Size to Reach USD 2.3 billion by 2031 at 4.6% CAGR"
query_embedding = encoder.encode(test_query)

query_embedding

NameError: name 'encoder' is not defined

# Черновики

РАБОЧИЙ ВАРИАНТ!!! Быстрый вариант загрузки данных в коллекцию c использование сплиттера с параметрами:

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30,
    )

def dataset_stream(df, splitter, gpu_batch_size=64):
    meta_cols = ["source_name", "title", "url", "published_at", "category"]
    
    batch_texts = []
    batch_payloads = []
    
    for row in df.itertuples(index=False):
        # Базовые метаданные текущей строки
        payload_base = {col: getattr(row, col) for col in meta_cols}
        
        # Режем текст переданным сплиттером
        chunks = splitter.split_text(str(row.full_content))
        
        for chunk in chunks:
            batch_texts.append(f"passage: {chunk}")
            # Добавляем сам чанк в метаданные, чтобы видеть его при поиске
            current_payload = payload_base.copy()
            current_payload["text"] = chunk
            batch_payloads.append(current_payload)
            
            # Если набрали батч для GPU — кодируем и отдаем
            if len(batch_texts) == gpu_batch_size:
                vectors = encoder.encode(batch_texts, convert_to_numpy=True)
                for v, p in zip(vectors, batch_payloads):
                    yield v, p
                batch_texts, batch_payloads = [], []
                
    # Доотправляем остатки
    if batch_texts:
        vectors = encoder.encode(batch_texts, convert_to_numpy=True)
        for v, p in zip(vectors, batch_payloads):
            yield v, p

dataset_streamer = dataset_stream(stratified_sample, text_splitter, gpu_batch_size=128)

client.upload_collection(
    collection_name=collection_name,
    vectors=(item[0] for item in dataset_streamer),
    payload=(item[1] for item in dataset_streamer),
    ids=None,
    batch_size=1000,
    parallel=1, # Only 1 GPU...
)

Загрузим и подготовим датасет:

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

tqdm.pandas()


def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=30,
    )
    return splitter.split_text(text)




dataset['full_content'] = dataset['full_content'].progress_apply(split_text)

dataset = dataset.explode('full_content').reset_index(drop=True)

dataset

  0%|          | 0/54834 [00:00<?, ?it/s]

,source_name,title,url,published_at,category,full_content
0,international business times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,nepal,Title: UN Chief Urges World To 'Stop The Madne...
1,international business times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,nepal,"the devastating impact of the phenomenon. ""The..."
2,international business times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,nepal,-- the ones here in the Himalayas supply fresh...
3,international business times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,nepal,Nepal. Glaciers in the wider Himalayan and Hin...
4,international business times,UN Chief Urges World To 'Stop The Madness' Of ...,https://www.ibtimes.com/un-chief-urges-world-s...,2023-10-30T10:12:35Z,nepal,"of the world's most important river systems, i..."
...,...,...,...,...,...,...
1259080,forbes,Tips For Investing In Short-Term Rentals In Dubai,https://www.forbes.com/sites/forbesbusinesscou...,2023-11-29T14:00:00Z,home,your property will appeal to. With plenty of l...
1259081,forbes,Tips For Investing In Short-Term Rentals In Dubai,https://www.forbes.com/sites/forbesbusinesscou...,2023-11-29T14:00:00Z,home,"the locations, markets and available services ..."
1259082,forbes,Tips For Investing In Short-Term Rentals In Dubai,https://www.forbes.com/sites/forbesbusinesscou...,2023-11-29T14:00:00Z,home,and your rates need to be competitive for your...
1259083,forbes,Tips For Investing In Short-Term Rentals In Dubai,https://www.forbes.com/sites/forbesbusinesscou...,2023-11-29T14:00:00Z,home,"pay rent, make requests and communicate. If yo..."


In [ ]:
from sentence_transformers import SentenceTransformer

batch = dataset[:10]["full_content"].to_list()

encoder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

embeddings = encoder.encode(batch)

print(embeddings.shape)

In [243]:
def product_of_odds(data):   # data - список целых чисел
    result = 1
    for i in data:
        if i % 2 == 1:
            result *= i

    return result

print(product_of_odds([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]))

945
